In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    groq_api_key=api_key,
)

response = llm.invoke("What is the capital of India?")
print(response.content)


c:\Users\dilji\Documents\AI-Engineer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The capital of India is New Delhi.


prompt template

In [3]:
from langchain_core.prompts import PromptTemplate

template = "Translate the following English text to Malayalam: {text}"
prompt = PromptTemplate.from_template(template)
final_prompt = prompt.format(text="Hello, how are you?")
response = llm.invoke(final_prompt)
print(response.content)

ഹലോ, എങ്ങനെയിരിക്കുന്നു? (Halo, engane irikkunnu?)


simple chain

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence
prompt = PromptTemplate.from_template("What is the capital of {country}?")
llm = llm
parser = StrOutputParser()
chain = RunnableSequence(prompt | llm | parser)
response = chain.invoke({"country": "France"})
print(response)

The capital of France is Paris.


In [ ]:
from langchain_core.runnables import RunnableMap 
chain = RunnableMap({
    "joke": PromptTemplate.from_template("Tell me a joke about {topic}.") | llm | parser,
    "fact": PromptTemplate.from_template("Tell me an interesting fact about {topic}.") | llm | parser,
})

result = chain.invoke({"topic": "space"})
print(result)

SyntaxError: invalid syntax (810482848.py, line 1)

runnable passthrough

In [13]:
from langchain_core.runnables import RunnablePassthrough
prompt = PromptTemplate.from_template("Summarize: {text}")
chain = RunnableMap({
    "original": RunnablePassthrough(),
    "summary": prompt | llm | parser,
})

result = chain.invoke({"text": "LangChain is a framework for developing applications powered by language models."})
print(result)

{'original': {'text': 'LangChain is a framework for developing applications powered by language models.'}, 'summary': 'LangChain is an open-source framework designed to simplify the process of building applications that utilize large language models (LLMs). It provides a set of tools and infrastructure to help developers create, deploy, and manage LLM-powered applications efficiently. With LangChain, developers can focus on building innovative applications without worrying about the underlying complexities of language model integration. The framework supports various LLMs and offers features such as data processing, model serving, and application templates to streamline the development process.'}


In [16]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://careers.nike.com/software-engineer-ii-itc/job/R-44289")
page_data = loader.load().pop().page_content
print(page_data)





















Software Engineer II, ITC










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



S

In [18]:
prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE :
    {page_data}
    ### INSTRUCTIONS :
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON format containing the 
    following keys: `role`, `experience`, `skills` and `description`.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE) :
    """
)

chain_extract = RunnableSequence(
    prompt_extract | llm | parser
)
result_extract = chain_extract.invoke({"page_data": page_data})
type(result_extract)

str

In [19]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()
json_res = json_parser.parse(result_extract)
print(json_res)

{'role': 'Software Engineer II, ITC', 'experience': '2+ years of hands-on industry software development experience', 'skills': ['front-end frameworks like React, Angular, or Vue.js', 'cloud architecture, modern DevOps, infrastructure as code, CI/CD and related tools', 'AWS products including Lambda, Step Functions, DynamoDB, Elasticsearch, s3', 'modern testing methodologies and frameworks such as Mocha, Jasmine and Jest', 'architectural design patterns and computer-science fundamentals', 'implementing and integrating AI, Machine Learning and related data solutions'], 'description': 'We’re looking for a Software Engineer to solve complex software engineering problems supporting Nike’s pursuit of delivering state of the art tools to our Consumer Product & Innovation community. The candidate needs to be highly collaborative with peers, productive in a fast-paced development environment and have depth of native cloud software engineering experience.'}


In [20]:
import pandas as pd
df = pd.read_csv("my_portfolio.csv")
print(df.head())

                           Techstack                                  Links
0            React, Node.js, MongoDB    https://example.com/react-portfolio
1           Angular,.NET, SQL Server  https://example.com/angular-portfolio
2  Vue.js, Ruby on Rails, PostgreSQL      https://example.com/vue-portfolio
3              Python, Django, MySQL   https://example.com/python-portfolio
4          Java, Spring Boot, Oracle     https://example.com/java-portfolio


In [21]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row['Techstack'],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [26]:
links = collection.query(query_texts=job['skills'], n_results=2).get("metadatas", [])
links

[[{'links': 'https://example.com/typescript-frontend-portfolio'},
  {'links': 'https://example.com/vue-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/vue-portfolio'}],
 [{'links': 'https://example.com/vue-portfolio'},
  {'links': 'https://example.com/full-stack-js-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}]]

In [25]:
job = json_res
job['skills']

['front-end frameworks like React, Angular, or Vue.js',
 'cloud architecture, modern DevOps, infrastructure as code, CI/CD and related tools',
 'AWS products including Lambda, Step Functions, DynamoDB, Elasticsearch, s3',
 'modern testing methodologies and frameworks such as Mocha, Jasmine and Jest',
 'architectural design patterns and computer-science fundamentals',
 'implementing and integrating AI, Machine Learning and related data solutions']

In [28]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION :
    {job_description}
    ### INSTRUCTIONS :
    You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
    the seamless integration of business processes through automated tools. 
    Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
    process optimization, cost reduction, and heightened overall efficiency. 
    Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
    in fulfilling their needs.
    Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
    Remember you are Mohan, BDE at AtliQ. 
    Do not provide a preamble.
    ### EMAIL (NO PREAMBLE):

    """
)
chain_email = RunnableSequence(
    prompt_email | llm | parser
)

res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res)

Subject: Expert Software Engineering Solutions for Nike's ITC Team

Dear Hiring Manager,

I came across the job description for a Software Engineer II, ITC at Nike, and I'm excited to introduce AtliQ, an AI & Software Consulting company that can help fulfill your software engineering needs. With our expertise in native cloud software engineering, we're confident that we can support Nike's pursuit of delivering state-of-the-art tools to the Consumer Product & Innovation community.

At AtliQ, we have a proven track record of empowering enterprises with tailored solutions, fostering scalability, process optimization, cost reduction, and heightened overall efficiency. Our team has hands-on experience with front-end frameworks like React, Angular, and Vue.js, as well as cloud architecture, modern DevOps, infrastructure as code, CI/CD, and related tools. We're well-versed in AWS products, including Lambda, Step Functions, DynamoDB, Elasticsearch, and S3.

Our expertise in modern testing meth